In [1]:
import os
import sys

os.environ["HADOOP_HOME"] = r"C:\tools\hadoop"
os.environ["PATH"] = r"C:\tools\hadoop\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import types
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.extraJavaOptions", "-Djava.library.path=C:/tools/hadoop/bin") \
    .getOrCreate()

In [3]:
# !curl -fLO https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

In [4]:
# !wc -l fhvhv_tripdata_2021-01.parquet

In [5]:
!uv add "pandas<3"
!uv add pyarrow

Resolved 101 packages in 0.75ms
Checked 9 packages in 0.44ms
Resolved 101 packages in 0.75ms
Checked 9 packages in 0.45ms


In [6]:
# pd.read_parquet("fhvhv_tripdata_2021-01.parquet").to_csv("fhvhv_tripdata_2021-01.csv", index=False)

In [7]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [8]:
# df.schema

In [9]:
# !head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [10]:
df_pandas = pd.read_csv('head.csv')

In [11]:
# df_pandas.dtypes

In [12]:
# spark.createDataFrame(df_pandas).show()

In [13]:
schema = schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [14]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [15]:
# df.head(10)

In [16]:
df = df.repartition(24)

In [17]:
df.write.parquet('fhvhv/2021/01/', mode='overwrite')

In [17]:
df = spark.read.parquet('fhvhv/2021/01/')

In [18]:
# df.printSchema()

In [19]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003').show() # .take(5) # head(5)

+---------------+-------------------+------------+------------+
|pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+---------------+-------------------+------------+------------+
|           NULL|2021-01-01 07:45:43|        NULL|        NULL|
|           NULL|2021-01-03 10:44:38|        NULL|        NULL|
|           NULL|2021-01-01 22:07:25|        NULL|        NULL|
|           NULL|2021-01-03 04:36:21|        NULL|        NULL|
|           NULL|2021-01-01 01:39:27|        NULL|        NULL|
|           NULL|2021-01-02 14:30:54|        NULL|        NULL|
|           NULL|2021-01-01 06:10:40|        NULL|        NULL|
|           NULL|2021-01-01 14:32:12|        NULL|        NULL|
|           NULL|2021-01-02 15:59:42|        NULL|        NULL|
|           NULL|2021-01-03 07:22:18|        NULL|        NULL|
|           NULL|2021-01-01 10:05:25|        NULL|        NULL|
|           NULL|2021-01-02 16:00:00|        NULL|        NULL|
|           NULL|2021-01-02 07:37:49|   

In [20]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [21]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [22]:
crazy_stuff('B02884')

's/b44'

In [25]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce|       NULL|  2021-01-02|        NULL|        NULL|
|  a/b40|       NULL|  2021-01-01|        NULL|        NULL|
|  e/9ce|       NULL|  2021-01-02|        NULL|        NULL|
|  a/b37|       NULL|  2021-01-03|        NULL|        NULL|
|  e/9ce|       NULL|  2021-01-01|        NULL|        NULL|
|  e/b30|       NULL|  2021-01-01|        NULL|        NULL|
|  e/b35|       NULL|  2021-01-03|        NULL|        NULL|
|  e/b32|       NULL|  2021-01-01|        NULL|        NULL|
|  e/9ce|       NULL|  2021-01-01|        NULL|        NULL|
|  e/b35|       NULL|  2021-01-02|        NULL|        NULL|
|  e/9ce|       NULL|  2021-01-02|        NULL|        NULL|
|  e/b30|       NULL|  2021-01-01|        NULL|        NULL|
|  e/a39|       NULL|  2021-01-01|        NULL|        NULL|
|  e/9ce|       NULL|  2

'python3'